# Tutorial 9.3 · Multi-Objective Power Splitter and Mode Converter (`ceviche_power_splitter` & `ceviche_mode_converter`)

**Chapter 9 — Inverse Design**

This tutorial presents Wakis's full-scale inverse-design results on two multi-objective
integrated-photonics benchmark challenges from `invrs-gym` (Schubert et al.,
arXiv:`2410.24132`; Schubert et al., *ACS Photonics* 2022):

1. **`ceviche_power_splitter`**: a compact $1.6\,\mu\text{m}\times 1.6\,\mu\text{m}$
   $1\times 2$ symmetric $50/50$ waveguide power splitter ($\text{TM}_0 \to \text{TM}_0 + \text{TM}_0$
   in the 2D $E_z$ polarization) operating across 6 wavelengths
   $\lambda \in \{1265, 1270, 1275, 1285, 1290, 1295\}\,\text{nm}$ on a $10\,\text{nm}$
   grid with an $8\,\text{px} = 80\,\text{nm}$ minimum solid/void feature length scale.
   Our Round-2 design achieves an official `ceviche` FDFD evaluation metric of
   **$+0.0682$** ($+0.068155$) at **`8 / 8 px`** ($96.88\%\text{--}97.59\%$ total
   guided transmission, $70.7\%$ of the $+0.0964$ lossless passivity ceiling),
   **beating the `invrs-gym` leaderboard best of $+0.0090$** ($+0.009037$ at `9 / 8 px`)
   by a factor of $7.5\times$ in normalized window margin.
2. **`ceviche_mode_converter`**: a $1.6\,\mu\text{m}\times 1.6\,\mu\text{m}$ waveguide
   spatial mode converter ($\text{TM}_0 \to \text{TM}_1$, mapping the fundamental even
   $E_z$ mode to the first-order odd $E_z$ mode) across the same 6 wavelengths and
   $8\,\text{px} = 80\,\text{nm}$ length-scale constraint. Our Round-2 design achieves
   an official evaluation metric of **$+0.3122$** at **`8 / 8 px`** ($92.52\%\text{--}93.40\%$
   mode conversion efficiency and $<-32.3\,\text{dB}$ reflection across all 6 channels),
   which satisfies all target windows and foundry rules but trails the `invrs-gym`
   leaderboard best of **$+0.8536$ at `8 / 8 px`** (and **$+0.9095$ at `6 / 6 px`**).
   We analyze the stored optimization traces, candidate pool, and DFT wavefields to
   diagnose exactly which target window and wavelength limit the score, separate
   back-reflection from cladding radiation loss, and formulate testable hypotheses for
   closing the gap.

### Learning goals

1. **Target-orthotope multi-objective evaluation metric (`eval_metric`)**: formulate the
   exact normalized signed distance to the multi-port, multi-wavelength S-parameter
   target orthotope $[l_j(\lambda_k), u_j(\lambda_k)]$ for both challenges and understand
   why a lossless $50/50$ splitter has a passivity ceiling of $\mathcal{M}_{\max} = +0.0964$
   whereas the mode converter has $\mathcal{M}_{\max} = +1.0$.
2. **Round-1 $\to$ Round-2 progression (enforcing foundry length scale by construction)**:
   see why Round-1 density + level-set runs violated the minimum feature size
   (`1 / 2 px` on the lightweight splitter at $+0.0669$; `2 / 3 px` and `1 / 1 px` on
   the lightweight mode converter at $-0.3101$ and $-3.8923$) and how Round-2's
   `FeasibleDensity` (`conic_filter` + Wang–Lazarov–Sigmund $\tanh$ projection +
   alternating `imageruler` circular-brush opening/closing repair + $\beta = 64$ STE)
   guarantees `8 / 8 px` ($80\,\text{nm}$) feasibility on every candidate.
3. **Robust multi-resolution objective ($40\,\text{nm} + 20\,\text{nm}$) & subpixel
   smoothing (`smoothing="tensor"` and SSP)**: understand how averaging the loss across
   $r = 1$ ($40\,\text{nm}$) and $r = 2$ ($20\,\text{nm}$) Wakis grids (`multi_resolution_loss`)
   and using `PaddedMaterialGrid(..., smoothing="tensor")` or subpixel-smoothed
   projection (`studies/ssp`) affect $10\,\text{nm}$ transferability (noting the
   2-to-4-seed pilot spread caveats).
4. **Honest diagnostic analysis of the mode-converter gap ($+0.3122$ vs. $+0.8536$)**:
   trace the bottleneck to the $\lambda_1 = 1265\,\text{nm}$ conversion floor
   ($|S_{21}|^2 = 0.9252 = -0.338\,\text{dB}$), $6.6\%\text{--}7.5\%$ cladding radiation
   loss (rather than reflection, which is $\le -32.3\,\text{dB}$), the Stage-1 $\to$
   Stage-2 morphological repair drop ($+0.4566 \to +0.2598$), and fixed-learning-rate
   STE boundary oscillations.
5. **Standalone Wakis re-simulation of the stored final designs**: rebuild the exact
   $344\times 280$ ($10\,\text{nm}$ pixel) geometry, waveguides, CPML, and polarized
   eigenmode ports from `fields_compact.npz` and `design.npy`, and verify that Wakis
   reproduces the stored modal S-parameters (`wakis_r1_modal_S2`) and dense 67-point
   spectra (`dense_S`).

---

## Physics, Evaluation Metric & Round-1 $\to$ Round-2 Pipeline

### 1. Multi-objective target-orthotope `eval_metric`

For a device with $N_p$ ports excited at port $1$ across $N_\lambda = 6$ wavelengths
$\lambda_k \in \{1265, 1270, 1275, 1285, 1290, 1295\}\,\text{nm}$, let
$T_{j}(\lambda_k) = |S_{j1}(\lambda_k)|^2$ denote the power fraction coupled into the
target waveguide eigenmode of port $j \in \{1, \dots, N_p\}$. Each challenge specifies
lower bounds $l_j(\lambda_k) \in [0, 1)$ and upper bounds $u_j(\lambda_k) \in (0, 1]$
defining a target orthotope (Schubert et al., arXiv:`2410.24132`):

- **`ceviche_power_splitter` ($N_p = 3$ ports, all $\text{TM}_0$ even mode)**:
  - Port 1 (input reflection $|S_{11}|^2$): $[l_1, u_1] = [0.0,\, 10^{-20/10}] = [0.0,\, 0.01]$ ($R_{\text{dB}} \le -20\,\text{dB}$).
  - Ports 2 & 3 (upper/lower output arms $|S_{21}|^2, |S_{31}|^2$):
    $[l_j, u_j] = [10^{-3.5/10},\, 1.0] \approx [0.446684,\, 1.0]$ ($T_{\text{dB}} \ge -3.5\,\text{dB}$).
- **`ceviche_mode_converter` ($N_p = 2$ ports: port 1 $\text{TM}_0$ even $\to$ port 2 $\text{TM}_1$ odd)**:
  - Port 1 (input $\text{TM}_0$ reflection $|S_{11}|^2$): $[l_1, u_1] = [0.0,\, 0.01]$ ($R_{\text{dB}} \le -20\,\text{dB}$).
  - Port 2 (output $\text{TM}_1$ conversion $|S_{21}|^2$):
    $[l_2, u_2] = [10^{-0.5/10},\, 1.0] \approx [0.891251,\, 1.0]$ ($T_{\text{dB}} \ge -0.5\,\text{dB}$).

Bounds at theoretical extrema ($l_j = 0$ or $u_j = 1$) are inactive as one-sided walls
but set the normalization width $w_j = u_j - l_j$. The per-channel margin and scalar
`eval_metric` $\mathcal{M}$ are:

$$
m_j(\lambda_k) = \min\!\left(
\begin{cases}
\dfrac{|S_{j1}(\lambda_k)|^2 - l_j}{u_j - l_j}, & l_j > 0 \\
+\infty, & l_j = 0
\end{cases},\;
\begin{cases}
\dfrac{u_j - |S_{j1}(\lambda_k)|^2}{u_j - l_j}, & u_j < 1 \\
+\infty, & u_j = 1
\end{cases}
\right), \qquad
\mathcal{M} = \min_{k, j} m_j(\lambda_k).
$$

Thus $\mathcal{M} > 0$ if and only if **every** port and wavelength channel lies inside
its specification window. For a passive, symmetric $1\times 2$ splitter
($|S_{21}|^2 = |S_{31}|^2 \le 0.5$), the theoretical lossless ceiling is
$\mathcal{M}_{\max}^{\text{splitter}} = (0.5 - 10^{-3.5/10}) / (1.0 - 10^{-3.5/10}) = +0.096358$.

### 2. Summary of Round-1, Round-2 & Leaderboard Results

| Challenge & Entry | Grid / Eval | Official `eval_metric` | Wakis `r=1` (`ceviche` / `modal`) | Min Solid / Void (`px`) | Target (`px`) | Status & Key Finding |
| :--- | :---: | :---: | :---: | :---: | :---: | :--- |
| **Power Splitter — Wakis Round-2 (`seed 1`)** | $10\,\text{nm}$ full | **$+0.0682$** ($+0.068155$) | $+0.0680$ / $+0.0679$ | **`8 / 8 px`** | `8 / 8 px` | **Beats leaderboard ($7.5\times$ margin); $70.7\%$ of lossless ceiling** |
| **Power Splitter — Wakis Round-2 (Rerun)** | $10\,\text{nm}$ full | **$+0.0676$** ($+0.067611$) | — | **`8 / 8 px`** | `8 / 8 px` | Re-scored in `leaderboard_rescore.json` |
| **Power Splitter — Leaderboard Best** | $10\,\text{nm}$ full | $+0.0090$ ($+0.009037$) | — | `9 / 8 px` | `8 / 8 px` | `240812_mfschubert_gym_levelset_example.csv` |
| **Power Splitter — Wakis Round-2 Pilot** | $40\,\text{nm}$ ($10\,\text{nm}$ chk) | $+0.052$ ($+0.045$) | — | **`3 / 3–4 / 4 px`** | `3 / 3 px` | $100\%$ length-scale pass on lightweight pilot |
| **Power Splitter — Wakis Round-1 (`seed 0 / 1`)** | $40\,\text{nm}$ ($10\,\text{nm}$ chk) | $+0.0669$ / $+0.0579$ ($+0.0634$ / $+0.0534$) | $+0.0673$ / $+0.0575$ | `1 / 2 px` | `3 / 3 px` | **Violated length scale** before Round-2 brush repair |
| **Mode Converter — Wakis Round-2 (`seed 1`)** | $10\,\text{nm}$ full | **$+0.3122$** ($+0.312213$) | $+0.3082$ / $+0.3079$ | **`8 / 8 px`** | `8 / 8 px` | Feasible; limited by $1265\,\text{nm}$ conversion ($92.5\%$, $\sim 7\%$ rad. loss) |
| **Mode Converter — Leaderboard Feasible** | $10\,\text{nm}$ full | **$+0.8536$** | — | `8 / 8 px` | `8 / 8 px` | Best feasible leaderboard reference ($\ge 98.4\%$ conversion) |
| **Mode Converter — Leaderboard Unconstrained** | $10\,\text{nm}$ full | $+0.9095$ | — | `6 / 6 px` | `8 / 8 px` | Violates $8\,\text{px}$ minimum width & spacing |
| **Mode Converter — Wakis Round-2 Pilot** | $40\,\text{nm}$ ($10\,\text{nm}$ chk) | $+0.381$ ($+0.315$) | — | **`3 / 3 px`** | `3 / 3 px` | $r=1+r=2$ multi-resolution pilot |
| **Mode Converter — Wakis Round-1 (`seed 0 / 1`)** | $40\,\text{nm}$ ($10\,\text{nm}$ chk) | $-0.3101$ / $-3.8923$ ($-0.9929$ / $-2.6557$) | $-0.3081$ / $-3.8951$ | `2 / 3` & `1 / 1 px` | `3 / 3 px` | **Failed $-20\,\text{dB}$ reflection** ($|S_{11}|^2 = 0.0131, 0.0489$) & length scale |

In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path
import time

import matplotlib

matplotlib.use("Agg")
from matplotlib import animation
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import c as C0
from scipy.ndimage import binary_closing, binary_erosion, binary_opening
from scipy.signal import convolve2d

import wakis
from wakis.inverse import AdjointSimulation, MaterialGrid
from wakis.mode_solver import ModeMonitor, ModeSource, solve_modes
from wakis.sources import CurrentSource, GaussianSource

TUTORIAL_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_DIR = TUTORIAL_DIR / "data"
FIG_DIR = TUTORIAL_DIR / "figures"
BEND_DATA_DIR = TUTORIAL_DIR.parent / "09-02_waveguide_bend" / "data"
LEADERBOARD_PATH = BEND_DATA_DIR / "leaderboard_rescore.json"
SSP_DATA_DIR = TUTORIAL_DIR.parents[2] / "studies" / "ssp" / "data"

TASKS: dict[str, str] = {
    "metrics_and_length_scale": "Verify exact orthotope eval_metric and length scales on stored Round-1/Round-2/leaderboard designs.",
    "wakis_resimulation": "Rebuild and re-simulate stored power splitter and mode converter designs with Wakis.",
    "gap_and_candidate_analysis": "Analyze the mode converter leaderboard gap, power budget, and Stage-1 -> Stage-2 repair drop.",
    "toy_optimization": "Run a pedagogical toy 2D mode-conversion adjoint optimization demo with wakis.inverse.",
    "optional_invrs_gym": "Optional live invrs-gym re-score if invrs_gym is installed.",
}

SPLITTER_LB_F32 = float(np.float32(10.0 ** (-3.5 / 10.0)))
MODECONV_LB_F32 = float(np.float32(10.0 ** (-0.5 / 10.0)))
REFL_UB_F32 = float(np.float32(10.0 ** (-20.0 / 10.0)))
SPLITTER_PASSIVITY_CEILING = (0.5 - SPLITTER_LB_F32) / (1.0 - SPLITTER_LB_F32)


def get_task_table(quick: bool = False) -> list[dict]:
    """Return the task table for Tutorial 9.3 with verified resolution metrics."""
    # At 10 nm grid (or 40 nm lightweight), minimum wavelength inside Si (n=3.5) is 1265/3.5 = 361.4 nm -> 36.1 cells/wl (10 nm)
    cpw_val = 28.0 / float(np.sqrt(6.0))  # 11.43 >= 10 cells per minimum wavelength
    return [
        {
            "index": 0,
            "name": "multi_objective_metrics_and_length_scale",
            "description": (
                "Verify exact invrs-gym target-orthotope eval_metric on stored Round-1, "
                "Round-2, and leaderboard S-parameters, plus circular-brush minimum solid/void "
                "length-scale measurement on round2_splitter and round2_mode_converter designs."
            ),
            "cpw": 36.14,
            "cells_across_feature": 8.0,
        },
        {
            "index": 1,
            "name": "wakis_resimulation_splitter_and_mode_converter",
            "description": (
                "Rebuild the simulation geometry, waveguide stubs, and polarized eigenmode ports "
                "from fields_compact.npz and design.npy, re-simulate both devices with Wakis, "
                "and compare against stored wakis_r1_modal_S2, official_S2, and dense_S spectra."
            ),
            "cpw": 36.14 if not quick else 10.0,
            "cells_across_feature": 8.0 if not quick else 4.0,
            "coarsen": 4 if quick else 1,
        },
        {
            "index": 2,
            "name": "mode_converter_gap_and_candidate_analysis",
            "description": (
                "Analyze why round2_mode_converter (+0.3122 at 8/8 px) trails the leaderboard "
                "(+0.8536 at 8/8 px, +0.9095 at 6/6 px) while round2_splitter (+0.0682 at 8/8 px) "
                "beats the leaderboard (+0.0090 at 9/8 px): wavelength bottlenecks, reflection vs "
                "radiation loss, STE candidate trajectory, and multi-resolution / SSP pilot data."
            ),
            "cpw": 36.14,
            "cells_across_feature": 8.0,
        },
        {
            "index": 3,
            "name": "pedagogical_toy_splitter_and_modeconv_optimization",
            "description": (
                "Pedagogical toy optimization (explicitly separate from the Round-2 invrs-gym "
                "benchmark runs): live Wakis AdjointSimulation + MaterialGrid gradient ascent "
                "for a compact 1x2 splitter and asymmetric mode converter with tensor smoothing."
            ),
            "cpw": cpw_val,
            "cells_across_feature": 6.0,
            "n_grid": 40 if quick else 48,
            "n_design": 16 if quick else 20,
            "Nt": 320 if quick else 420,
            "n_iters": 8 if quick else 14,
        },
        {
            "index": 4,
            "name": "optional_invrs_gym_rescore",
            "description": (
                "Optional official invrs-gym / invrs-wakis challenge evaluation if installed "
                "in the Python environment."
            ),
            "cpw": 36.14,
            "cells_across_feature": 8.0,
        },
    ]


def compute_ceviche_eval_metric(
    s2: np.ndarray,
    lower_bounds: np.ndarray,
    upper_bounds: np.ndarray,
) -> tuple[float, np.ndarray]:
    """Compute the exact invrs-gym target-orthotope evaluation metric and per-channel margins.

    Parameters
    ----------
    s2 : np.ndarray
        Power transmission/reflection coefficients ``|S_{j1}(lambda_k)|^2`` of shape
        ``(n_wl, n_ports)`` or ``(n_wl, 1, n_ports)`` [dimensionless].
    lower_bounds : np.ndarray
        Target lower bounds ``l_j(lambda_k)`` broadcastable to ``(n_wl, n_ports)`` [dimensionless].
    upper_bounds : np.ndarray
        Target upper bounds ``u_j(lambda_k)`` broadcastable to ``(n_wl, n_ports)`` [dimensionless].

    Returns
    -------
    eval_metric : float
        Worst-case normalized signed distance ``min_{k,j} m_j(lambda_k)`` [dimensionless].
    margins : np.ndarray
        Channel margin array ``m_j(lambda_k)`` of shape ``(n_wl, n_ports)`` [dimensionless].
    """
    arr = np.asarray(s2, dtype=np.float64)
    if arr.ndim == 3:
        arr = arr[:, 0, :]
    lb = np.broadcast_to(np.asarray(lower_bounds, dtype=np.float64), arr.shape)
    ub = np.broadcast_to(np.asarray(upper_bounds, dtype=np.float64), arr.shape)
    width = ub - lb
    m_lo = np.where(lb > 0.0, (arr - lb) / width, np.inf)
    m_hi = np.where(ub < 1.0, (ub - arr) / width, np.inf)
    margins = np.minimum(m_lo, m_hi)
    return float(np.min(margins)), margins


def compute_splitter_eval_metric(s2: np.ndarray) -> tuple[float, np.ndarray]:
    """Evaluate ``ceviche_power_splitter`` orthotope metric (port 0 <= -20 dB; ports 1,2 >= -3.5 dB)."""
    lb = np.array([[0.0, SPLITTER_LB_F32, SPLITTER_LB_F32]], dtype=np.float64)
    ub = np.array([[REFL_UB_F32, 1.0, 1.0]], dtype=np.float64)
    return compute_ceviche_eval_metric(s2, lb, ub)


def compute_modeconv_eval_metric(s2: np.ndarray) -> tuple[float, np.ndarray]:
    """Evaluate ``ceviche_mode_converter`` orthotope metric (port 0 <= -20 dB; port 1 >= -0.5 dB)."""
    lb = np.array([[0.0, MODECONV_LB_F32]], dtype=np.float64)
    ub = np.array([[REFL_UB_F32, 1.0]], dtype=np.float64)
    return compute_ceviche_eval_metric(s2, lb, ub)


def circular_brush(diameter_px: int) -> np.ndarray:
    """Create an imageruler-style circular structuring element of diameter ``diameter_px`` [px]."""
    d = max(1, int(diameter_px))
    if d == 1:
        return np.ones((1, 1), dtype=bool)
    # Match imageruler's discrete Euclidean disk kernel for even and odd diameters
    coords = np.arange(d, dtype=np.float64) - 0.5 * (d - 1)
    xx, yy = np.meshgrid(coords, coords, indexing="ij")
    return (np.hypot(xx, yy) <= 0.5 * d + 1e-9).astype(bool)


def measure_length_scales_px(binary_arr: np.ndarray, max_px: int = 12, periodic: tuple[bool, bool] = (False, False)) -> tuple[int, int]:
    """Measure minimum solid width and minimum void spacing [px] via imageruler-style morphology.

    When ``imageruler`` is not installed in the offline environment, this function
    evaluates circular brush opening/closing on 4-connected interior pixels
    (matching ``imageruler``'s ``length_scale_violations_solid`` definition and
    reproducing the stored ``length_scale_leaderboard_px`` values).
    """
    b2 = np.asarray(binary_arr[:, :, 0] if binary_arr.ndim == 3 else binary_arr) >= 0.5
    if np.all(b2) or not np.any(b2):
        return max_px, max_px
    cross = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)
    int_solid = binary_erosion(b2, structure=cross, border_value=1)
    int_void = binary_erosion(~b2, structure=cross, border_value=1)
    pad = max(4, int(max_px) + 2)
    b_pad = np.pad(b2, ((pad, pad), (0, 0)), mode="wrap" if periodic[0] else "edge")
    b_pad = np.pad(b_pad, ((0, 0), (pad, pad)), mode="wrap" if periodic[1] else "edge")

    min_solid = 1
    for d in range(2, max_px + 1):
        brush = circular_brush(d)
        opened = binary_opening(b_pad, structure=brush)[pad:-pad, pad:-pad]
        if int(np.sum((opened != b2) & int_solid)) == 0:
            min_solid = d
        else:
            break

    min_void = 1
    for d in range(2, max_px + 1):
        brush = circular_brush(d)
        closed = binary_closing(b_pad, structure=brush)[pad:-pad, pad:-pad]
        if int(np.sum((closed != b2) & int_void)) == 0:
            min_void = d
        else:
            break

    return int(min_solid), int(min_void)


def _z_fraction(mode) -> float:
    """Fraction of transverse electric energy carried by the Ez component."""
    axis = {"b": mode.axes[1], "c": mode.axes[2]}
    ez = sum(np.sum(np.abs(v) ** 2) for k, v in mode.e.items() if axis[k] == "z")
    total = sum(np.sum(np.abs(v) ** 2) for v in mode.e.values())
    return float(ez / total)


def polarized_modes(solver, frequency: float, center, size, n_modes: int, polarization: str = "Ez"):
    """Return guided eigenmodes of a single polarization family sorted by decreasing effective index."""
    want_ez = polarization == "Ez"
    modes = solve_modes(solver, frequency, center, size, n_modes=2 * n_modes + 2)
    picked = [m for m in modes if (_z_fraction(m) > 0.5) == want_ez]
    return picked[:n_modes]


class PolarizedModeSource(ModeSource):
    """Waveguide mode source selecting modes within a single 2D polarization family."""

    def __init__(self, *args, polarization: str = "Ez", **kwargs):
        super().__init__(*args, **kwargs)
        self.polarization = polarization
        self.family_index = self.mode_index

    def _initialize(self, solver):
        n = self.family_index + 1
        family = polarized_modes(
            solver, self.mode_frequency, self.center, self.size, n, self.polarization
        )
        if len(family) < n:
            raise ValueError(f"{self.polarization} mode {n} does not propagate.")
        target = family[self.family_index]
        modes = solve_modes(
            solver, self.mode_frequency, self.center, self.size, n_modes=2 * n + 2
        )
        self.mode_index = int(np.argmin([abs(m.beta - target.beta) for m in modes]))
        super()._initialize(solver)


class PolarizedModeMonitor(ModeMonitor):
    """Waveguide mode monitor selecting modes within a single 2D polarization family."""

    def __init__(self, *args, polarization: str = "Ez", **kwargs):
        super().__init__(*args, **kwargs)
        self.polarization = polarization

    def _solve_modes(self):
        def family(f):
            return polarized_modes(
                self.solver, f, self.center, self.size, self.n_modes, self.polarization
            )

        if self.mode_frequency is not None:
            self.modes = [family(self.mode_frequency)] * self.frequencies.size
        else:
            self.modes = [family(f) for f in self.frequencies]


class _DummyDesign:
    """Zero-parameter design placeholder for pure forward AdjointSimulation runs."""

    shape = (1,)
    flat_idx = np.zeros(0, dtype=int)

    def apply(self, solver, rho):
        pass

    def dispersive_group(self, solver):
        return None

    def network_of(self, solver):
        return None


def simulate_ceviche_challenge_wakis(
    fields_npz_path: Path,
    design_npy_path: Path,
    eval_wl_um: np.ndarray,
    coarsen: int = 1,
    run_time_factor: float = 1.0,
) -> dict:
    """Rebuild and simulate a stored ceviche challenge design with Wakis alone.

    Parameters
    ----------
    fields_npz_path : Path
        Path to ``fields_compact.npz`` containing ``eps``, ``design_box``, ``ports``, and ``pml``.
    design_npy_path : Path
        Path to ``design.npy`` (160x100 design density in [0, 1]).
    eval_wl_um : np.ndarray
        Wavelengths [um] at which to extract complex modal S-parameters.
    coarsen : int
        Spatial grid coarsening factor relative to the native 10 nm grid (1 = native 10 nm, 2 = 20 nm, 4 = 40 nm).
    run_time_factor : float
        Multiplier on the physical propagation time window.

    Returns
    -------
    dict
        Dictionary with ``s_eval``, ``s2_eval``, ``s_dense``, ``s2_dense``, ``dense_wl_um``, ``nx``, ``ny``, ``n_steps``.
    """
    from scipy.constants import epsilon_0 as EPS_0

    fc = np.load(fields_npz_path)
    design = np.load(design_npy_path)
    design_2d = design[:, :, 0] if design.ndim == 3 else design
    eps_raw = fc["eps"].astype(np.float64) * float(fc["eps_scale"])
    # Native 10 nm grid has 344x280 cells (fields_compact.npz stores 345x281 node values)
    eps_full = eps_raw[:344, :280]
    x0_um, y0_um, x1_um, y1_um = fc["design_um"].astype(np.float64)
    ports = fc["ports"].astype(np.float64)
    pml_px = int(round(float(fc["pml_um"]) / 0.01))
    dense_wl_um = fc["dense_wavelengths_nm"].astype(np.float64) * 1e-3

    nx0, ny0 = eps_full.shape
    dx0 = 10e-9  # 10 nm native resolution [m]
    lx, ly = nx0 * dx0, ny0 * dx0

    # Rebuild permittivity map from background + design density (eps_min=2.085136, eps_max=12.1104)
    eps_bg = eps_full.copy()
    i0, i1 = int(round(x0_um / 0.01)), int(round(x1_um / 0.01))
    j0, j1 = int(round(y0_um / 0.01)), int(round(y1_um / 0.01))
    if i1 - i0 == design_2d.shape[0] and j1 - j0 == design_2d.shape[1]:
        eps_bg[i0:i1, j0:j1] = 2.085136 + (12.1104 - 2.085136) * np.clip(design_2d, 0.0, 1.0)

    if coarsen > 1:
        nx = nx0 // coarsen
        ny = ny0 // coarsen
        eps_sim = eps_bg.reshape(nx, coarsen, ny, coarsen).mean(axis=(1, 3))
        dx = dx0 * coarsen
        npml = max(8, pml_px // coarsen)
    else:
        nx, ny = nx0, ny0
        eps_sim = eps_bg
        dx = dx0
        npml = pml_px

    freqs = C0 / (eval_wl_um * 1e-6)
    dense_freqs = C0 / (dense_wl_um * 1e-6)
    all_freqs = np.concatenate([freqs, dense_freqs])
    f0 = float(np.mean(freqs))
    df = max(float(np.ptp(all_freqs)) * 0.75, 0.06 * f0)

    def _make_solver(eps_2d: np.ndarray):
        grid = wakis.GridFIT3D(
            0.0, lx, 0.0, ly, -0.5 * dx, 0.5 * dx, nx, ny, 1, verbose=0
        )
        solver = wakis.SolverFIT3D(
            grid,
            bc_low=["cpml", "cpml", "periodic"],
            bc_high=["cpml", "cpml", "periodic"],
            n_pml=npml,
            verbose=0,
        )
        eps_pad = np.pad(eps_2d, ((0, 1), (0, 1)), mode="edge")
        target_shape = solver.ieps.to_matrix("z").shape
        ieps = np.broadcast_to(1.0 / (eps_pad[: target_shape[0], : target_shape[1], None] * EPS_0), target_shape).copy()
        for c in ("x", "y", "z"):
            solver.ieps.from_matrix(ieps, c)
        solver.update_tensors("ieps")
        return solver

    is_modeconv = len(ports) == 2
    p0 = ports[0]
    offset_m = 5.0 * dx
    src_x = float(p0[0] * 1e-6 + p0[4] * offset_m)
    src_y = float(p0[1] * 1e-6)
    src_h = float(p0[2] * 1e-6)
    src_mode = 0
    src_dir = "+" if p0[4] > 0 else "-"

    src = PolarizedModeSource(
        GaussianSource(f0, df),
        direction=src_dir,
        center=(src_x, src_y, 0.0),
        size=(0.0, src_h, dx),
        mode_index=src_mode,
        mode_frequency=f0,
        polarization="Ez",
    )

    monitors = {}
    for j, p in enumerate(ports):
        mx = float(p[0] * 1e-6)
        my = float(p[1] * 1e-6)
        mh = float(p[2] * 1e-6)
        mmode = 1 if (is_modeconv and j == 1) else 0
        monitors[f"p{j}"] = PolarizedModeMonitor(
            all_freqs,
            center=(mx, my, 0.0),
            size=(0.0, mh, dx),
            n_modes=mmode + 1,
            mode_frequency=f0,
            polarization="Ez",
        )

    probe = _make_solver(eps_sim)
    dt = float(probe.dt)
    n_max = float(np.sqrt(np.max(eps_sim)))
    t_total = run_time_factor * (6.0 / df + 2.5 * max(lx, ly) * n_max / C0)
    n_steps = int(np.ceil(t_total / dt))

    # Straight input waveguide calibration run for complex incident mode amplitude a+(omega)
    ix_port0 = int(np.clip(round(p0[0] * 1e-6 / dx), 0, nx - 1))
    eps_ref_2d = np.broadcast_to(eps_sim[ix_port0 : ix_port0 + 1, :], (nx, ny)).copy()
    src_ref = PolarizedModeSource(
        GaussianSource(f0, df),
        direction=src_dir,
        center=(src_x, src_y, 0.0),
        size=(0.0, src_h, dx),
        mode_index=src_mode,
        mode_frequency=f0,
        polarization="Ez",
    )
    mon_ref = PolarizedModeMonitor(
        all_freqs,
        center=(float(p0[0] * 1e-6), src_y, 0.0),
        size=(0.0, src_h, dx),
        n_modes=src_mode + 1,
        mode_frequency=f0,
        polarization="Ez",
    )
    t_ref = run_time_factor * (6.0 / df + 1.5 * lx * n_max / C0)
    n_steps_ref = min(n_steps, int(np.ceil(t_ref / dt)))
    sim_ref = AdjointSimulation(lambda: _make_solver(eps_ref_2d), [src_ref], {"p0": mon_ref}, _DummyDesign(), n_steps_ref)
    sim_ref.forward(np.zeros(1), record=False)
    a_inc, b_inc = sim_ref._monitors["p0"].get_amplitudes(mode_index=src_mode)
    inc = a_inc if p0[4] > 0 else b_inc

    sim_full = AdjointSimulation(lambda: _make_solver(eps_sim), [src], monitors, _DummyDesign(), n_steps)
    sim_full.forward(np.zeros(1), record=False)

    n_eval = len(freqs)
    n_ports = len(ports)
    s_all = np.zeros((len(all_freqs), n_ports), dtype=np.complex128)
    for j, p in enumerate(ports):
        mmode = 1 if (is_modeconv and j == 1) else 0
        a_p, b_p = sim_full._monitors[f"p{j}"].get_amplitudes(mode_index=mmode)
        outgoing = a_p if p[4] < 0 else b_p
        s_all[:, j] = outgoing / inc

    s_eval = s_all[:n_eval, :]
    s_dense = s_all[n_eval:, :]
    return {
        "s_eval": s_eval,
        "s2_eval": np.abs(s_eval) ** 2,
        "s_dense": s_dense,
        "s2_dense": np.abs(s_dense) ** 2,
        "dense_wl_um": dense_wl_um,
        "nx": nx,
        "ny": ny,
        "n_steps": n_steps,
    }


def run_task_0_metrics_and_length_scale() -> dict:
    """Task 0: Verify multi-objective eval metrics, Round-1 vs Round-2 progression, and length scales."""
    sp_summary = json.loads((DATA_DIR / "round2_splitter" / "summary.json").read_text())
    mc_summary = json.loads((DATA_DIR / "round2_mode_converter" / "summary.json").read_text())
    r1_sp = json.loads(
        (DATA_DIR / "round1" / "20261018-0430-invrs-ceviche__shard_4__ceviche_lightweight_power_splitter_seed0__summary.json").read_text()
    )
    r1_mc = json.loads(
        (DATA_DIR / "round1" / "20261018-0430-invrs-ceviche__shard_2__ceviche_lightweight_mode_converter_seed0__summary.json").read_text()
    )
    lb_data = json.loads(LEADERBOARD_PATH.read_text())

    # 1. Verify exact orthotope eval_metric on Splitter
    sp_off_s2 = np.asarray(sp_summary["official_S2"], dtype=np.float64)
    sp_est_s2 = np.asarray(sp_summary["wakis_r1_ceviche_estimator_S2"], dtype=np.float64)
    sp_mod_s2 = np.asarray(sp_summary["wakis_r1_modal_S2"], dtype=np.float64)

    sp_off_eval, sp_off_margins = compute_splitter_eval_metric(sp_off_s2)
    sp_est_eval, sp_est_margins = compute_splitter_eval_metric(sp_est_s2)
    sp_mod_eval, sp_mod_margins = compute_splitter_eval_metric(sp_mod_s2)

    # 2. Verify exact orthotope eval_metric on Mode Converter
    mc_off_s2 = np.asarray(mc_summary["official_S2"], dtype=np.float64)
    mc_est_s2 = np.asarray(mc_summary["wakis_r1_ceviche_estimator_S2"], dtype=np.float64)
    mc_mod_s2 = np.asarray(mc_summary["wakis_r1_modal_S2"], dtype=np.float64)

    mc_off_eval, mc_off_margins = compute_modeconv_eval_metric(mc_off_s2)
    mc_est_eval, mc_est_margins = compute_modeconv_eval_metric(mc_est_s2)
    mc_mod_eval, mc_mod_margins = compute_modeconv_eval_metric(mc_mod_s2)

    # 3. Measure length scales on the binary designs
    sp_design = np.load(DATA_DIR / "round2_splitter" / "design.npy")
    mc_design = np.load(DATA_DIR / "round2_mode_converter" / "design.npy")
    sp_ls_measured = measure_length_scales_px(sp_design)
    mc_ls_measured = measure_length_scales_px(mc_design)

    # 4. Extract leaderboard best for splitter
    lb_sp_entries = [
        e for e in lb_data
        if e["challenge"] == "ceviche_power_splitter" and not str(e["file"]).startswith("ours_")
    ]
    lb_sp_positive = [e for e in lb_sp_entries if e["eval_metric"] > 0]
    lb_sp_best = max(lb_sp_entries, key=lambda e: e["eval_metric"])

    out = {
        "splitter": {
            "official_eval_stored": float(sp_summary["official_eval_metric"]),
            "official_eval_recomputed": sp_off_eval,
            "official_eval_abs_diff": abs(sp_off_eval - float(sp_summary["official_eval_metric"])),
            "wakis_estimator_eval_stored": float(sp_summary["wakis_r1_ceviche_estimator_eval_metric"]),
            "wakis_estimator_eval_recomputed": sp_est_eval,
            "wakis_modal_eval_stored": float(sp_summary["wakis_r1_modal_eval_metric"]),
            "wakis_modal_eval_recomputed": sp_mod_eval,
            "official_margins": sp_off_margins.tolist(),
            "length_scale_stored_px": sp_summary["length_scale_leaderboard_px"],
            "length_scale_measured_px": list(sp_ls_measured),
            "round1_best_official_eval": float(r1_sp["official_eval_metric"]),
            "round1_best_length_scale_px": r1_sp["length_scale_solid_void_px"],
            "leaderboard_best_eval": float(lb_sp_best["eval_metric"]),
            "leaderboard_best_file": lb_sp_best["file"],
            "leaderboard_best_length_scale_px": [int(lb_sp_best["minimum_width"]), int(lb_sp_best["minimum_spacing"])],
            "leaderboard_num_entries": len(lb_sp_entries),
            "leaderboard_num_positive": len(lb_sp_positive),
        },
        "mode_converter": {
            "official_eval_stored": float(mc_summary["official_eval_metric"]),
            "official_eval_recomputed": mc_off_eval,
            "official_eval_abs_diff": abs(mc_off_eval - float(mc_summary["official_eval_metric"])),
            "wakis_estimator_eval_stored": float(mc_summary["wakis_r1_ceviche_estimator_eval_metric"]),
            "wakis_estimator_eval_recomputed": mc_est_eval,
            "wakis_modal_eval_stored": float(mc_summary["wakis_r1_modal_eval_metric"]),
            "wakis_modal_eval_recomputed": mc_mod_eval,
            "official_margins": mc_off_margins.tolist(),
            "length_scale_stored_px": mc_summary["length_scale_leaderboard_px"],
            "length_scale_measured_px": list(mc_ls_measured),
            "round1_best_official_eval": float(r1_mc["official_eval_metric"]),
            "round1_best_length_scale_px": r1_mc["length_scale_solid_void_px"],
            "leaderboard_reported_6px": 0.9095,
            "leaderboard_reported_8px": 0.8536,
        },
    }
    (DATA_DIR / "metrics_and_length_scale_summary.json").write_text(json.dumps(out, indent=2))
    print(
        f"[Task 0] Splitter official_eval={sp_off_eval:+.6f} (diff={out['splitter']['official_eval_abs_diff']:.2e}), "
        f"LS={sp_ls_measured} px (stored {sp_summary['length_scale_leaderboard_px']})"
    )
    print(
        f"[Task 0] ModeConv official_eval={mc_off_eval:+.6f} (diff={out['mode_converter']['official_eval_abs_diff']:.2e}), "
        f"LS={mc_ls_measured} px (stored {mc_summary['length_scale_leaderboard_px']})"
    )
    return out


def run_task_1_wakis_resimulation(mode: str = "quick") -> dict:
    """Task 1: Re-simulate stored final designs with Wakis and compare against stored spectra."""
    sp_summary = json.loads((DATA_DIR / "round2_splitter" / "summary.json").read_text())
    mc_summary = json.loads((DATA_DIR / "round2_mode_converter" / "summary.json").read_text())
    sp_fc = np.load(DATA_DIR / "round2_splitter" / "fields_compact.npz")
    mc_fc = np.load(DATA_DIR / "round2_mode_converter" / "fields_compact.npz")
    eval_wl = np.array([1.265, 1.270, 1.275, 1.285, 1.290, 1.295], dtype=np.float64)

    resim_npz_path = DATA_DIR / "wakis_resim_full.npz"
    if mode == "full" or not resim_npz_path.exists():
        print("[Task 1] Running full-resolution (r=1, 10 nm, 344x280) Wakis re-simulations...")
        sp_res = simulate_ceviche_challenge_wakis(
            DATA_DIR / "round2_splitter" / "fields_compact.npz",
            DATA_DIR / "round2_splitter" / "design.npy",
            eval_wl,
            coarsen=1,
            run_time_factor=1.0,
        )
        mc_res = simulate_ceviche_challenge_wakis(
            DATA_DIR / "round2_mode_converter" / "fields_compact.npz",
            DATA_DIR / "round2_mode_converter" / "design.npy",
            eval_wl,
            coarsen=1,
            run_time_factor=1.0,
        )
        np.savez_compressed(
            resim_npz_path,
            wavelengths_nm=eval_wl * 1000.0,
            dense_wavelengths_nm=sp_res["dense_wl_um"] * 1000.0,
            splitter_s2_6=sp_res["s2_eval"],
            splitter_dense_S=sp_res["s_dense"].astype(np.complex64),
            splitter_s2_dense=sp_res["s2_dense"],
            modeconv_s2_6=mc_res["s2_eval"],
            modeconv_dense_S=mc_res["s_dense"].astype(np.complex64),
            modeconv_s2_dense=mc_res["s2_dense"],
        )

    # In --quick mode, run a fast coarsened live Wakis sanity run (~6 s) and verify full-res parity from wakis_resim_full.npz
    if mode == "quick":
        sp_quick = simulate_ceviche_challenge_wakis(
            DATA_DIR / "round2_splitter" / "fields_compact.npz",
            DATA_DIR / "round2_splitter" / "design.npy",
            eval_wl,
            coarsen=4,
            run_time_factor=0.35,
        )
        mc_quick = simulate_ceviche_challenge_wakis(
            DATA_DIR / "round2_mode_converter" / "fields_compact.npz",
            DATA_DIR / "round2_mode_converter" / "design.npy",
            eval_wl,
            coarsen=4,
            run_time_factor=0.35,
        )
        quick_steps = (sp_quick["n_steps"], mc_quick["n_steps"])
    else:
        quick_steps = (0, 0)

    resim = np.load(resim_npz_path)
    sp_mod_stored = np.asarray(sp_summary["wakis_r1_modal_S2"], dtype=np.float64)[:, 0, :]
    mc_mod_stored = np.asarray(mc_summary["wakis_r1_modal_S2"], dtype=np.float64)[:, 0, :]
    sp_dense_S = (sp_fc["dense_S_re"].astype(np.float64) + 1j * sp_fc["dense_S_im"].astype(np.float64)) * float(sp_fc["dense_S_scale"])
    mc_dense_S = (mc_fc["dense_S_re"].astype(np.float64) + 1j * mc_fc["dense_S_im"].astype(np.float64)) * float(mc_fc["dense_S_scale"])
    sp_dense_stored = np.abs(sp_dense_S) ** 2
    mc_dense_stored = np.abs(mc_dense_S) ** 2

    sp_max_eval_diff = float(np.max(np.abs(resim["splitter_s2_6"] - sp_mod_stored)))
    mc_max_eval_diff = float(np.max(np.abs(resim["modeconv_s2_6"] - mc_mod_stored)))
    sp_max_dense_diff = float(np.max(np.abs(resim["splitter_s2_dense"] - sp_dense_stored)))
    mc_max_dense_diff = float(np.max(np.abs(resim["modeconv_s2_dense"] - mc_dense_stored)))

    sp_resim_eval, _ = compute_splitter_eval_metric(resim["splitter_s2_6"])
    mc_resim_eval, _ = compute_modeconv_eval_metric(resim["modeconv_s2_6"])

    out = {
        "mode": mode,
        "quick_coarsened_steps": list(quick_steps),
        "splitter_max_abs_diff_vs_wakis_r1_modal_S2": sp_max_eval_diff,
        "splitter_max_abs_diff_vs_dense_S2": sp_max_dense_diff,
        "splitter_resim_modal_eval_metric": sp_resim_eval,
        "splitter_stored_modal_eval_metric": float(sp_summary["wakis_r1_modal_eval_metric"]),
        "modeconv_max_abs_diff_vs_wakis_r1_modal_S2": mc_max_eval_diff,
        "modeconv_max_abs_diff_vs_dense_S2": mc_max_dense_diff,
        "modeconv_resim_modal_eval_metric": mc_resim_eval,
        "modeconv_stored_modal_eval_metric": float(mc_summary["wakis_r1_modal_eval_metric"]),
    }
    (DATA_DIR / "wakis_resim_comparison.json").write_text(json.dumps(out, indent=2))
    print(
        f"[Task 1] Splitter Wakis re-sim max|dS2|={sp_max_eval_diff:.2e} (eval={sp_resim_eval:+.6f}), "
        f"dense max|dS2|={sp_max_dense_diff:.2e}"
    )
    print(
        f"[Task 1] ModeConv Wakis re-sim max|dS2|={mc_max_eval_diff:.2e} (eval={mc_resim_eval:+.6f}), "
        f"dense max|dS2|={mc_max_dense_diff:.2e}"
    )
    return out


def run_task_2_gap_and_candidate_analysis() -> dict:
    """Task 2: Analyze candidate selection, Stage-1 vs Stage-2 drop, and the Mode Converter leaderboard gap."""
    sp_summary = json.loads((DATA_DIR / "round2_splitter" / "summary.json").read_text())
    mc_summary = json.loads((DATA_DIR / "round2_mode_converter" / "summary.json").read_text())
    sp_snaps = np.load(DATA_DIR / "round2_splitter" / "snapshots_thinned.npz")
    mc_snaps = np.load(DATA_DIR / "round2_mode_converter" / "snapshots_thinned.npz")
    eval_wl_nm = [1265, 1270, 1275, 1285, 1290, 1295]

    mc_off_s2 = np.asarray(mc_summary["official_S2"], dtype=np.float64)[:, 0, :]
    _, mc_margins = compute_modeconv_eval_metric(mc_off_s2)

    # Decompose power budget for mode converter at each challenge wavelength:
    # Reflection R = |S_11|^2, Conversion T_TE1 = |S_21|^2, Unaccounted (Radiation + unconverted TE0) = 1 - R - T_TE1
    mc_wavelength_breakdown = []
    for k, wl in enumerate(eval_wl_nm):
        r_val = float(mc_off_s2[k, 0])
        t_val = float(mc_off_s2[k, 1])
        unacc = float(1.0 - r_val - t_val)
        m_refl = float(mc_margins[k, 0])
        m_trans = float(mc_margins[k, 1])
        limiting = "transmission_TE1" if m_trans <= m_refl else "reflection_TE0"
        mc_wavelength_breakdown.append(
            {
                "wavelength_nm": wl,
                "reflection_S11_sq": r_val,
                "reflection_dB": float(10.0 * np.log10(max(r_val, 1e-12))),
                "transmission_TE1_S21_sq": t_val,
                "transmission_TE1_dB": float(10.0 * np.log10(max(t_val, 1e-12))),
                "radiation_or_TE0_leakage": unacc,
                "margin_reflection": m_refl,
                "margin_transmission": m_trans,
                "limiting_channel": limiting,
            }
        )

    # Analyze Stage 1 (density_beta4..32) vs Stage 2 (ste_beta64) in snapshots_thinned.npz
    mc_iters = mc_snaps["kept_index"].astype(int)
    mc_stages = mc_snaps["stage"].astype(str)
    mc_evals = mc_snaps["eval_metric"].astype(float)
    s1_mask = mc_stages != "ste_beta64"
    s2_mask = mc_stages == "ste_beta64"
    mc_stage1_peak_idx = int(np.argmax(mc_evals[s1_mask]))
    mc_stage1_peak_iter = int(mc_iters[s1_mask][mc_stage1_peak_idx])
    mc_stage1_peak_eval = float(mc_evals[s1_mask][mc_stage1_peak_idx])
    mc_stage2_entry_eval = float(mc_evals[s2_mask][0])
    mc_stage2_peak_eval = float(np.max(mc_evals[s2_mask]))

    out = {
        "mode_converter_wavelength_breakdown": mc_wavelength_breakdown,
        "worst_wavelength_nm": 1265,
        "worst_channel": "transmission_TE1",
        "worst_margin": float(np.min(mc_margins)),
        "stage1_peak_iter": mc_stage1_peak_iter,
        "stage1_peak_surrogate_eval": mc_stage1_peak_eval,
        "stage2_entry_surrogate_eval": mc_stage2_entry_eval,
        "stage2_peak_surrogate_eval": mc_stage2_peak_eval,
        "candidates_mode_converter": mc_summary["candidates"],
        "candidates_splitter": sp_summary["candidates"],
    }
    (DATA_DIR / "gap_and_candidate_analysis.json").write_text(json.dumps(out, indent=2))
    print(
        f"[Task 2] ModeConv Stage-1 peak eval={mc_stage1_peak_eval:+.4f} (iter {mc_stage1_peak_iter}) -> "
        f"Stage-2 entry={mc_stage2_entry_eval:+.4f}, worst channel={out['worst_channel']} at {out['worst_wavelength_nm']} nm"
    )
    return out


def run_task_3_pedagogical_toy_optimization(mode: str = "quick") -> dict:
    """Task 3: Pedagogical toy 2D mode-conversion adjoint demo using wakis.inverse (separate from invrs-gym)."""
    cfg = get_task_table(quick=(mode == "quick"))[3]
    n_iters = int(cfg["n_iters"])
    n_steps = int(cfg["Nt"])
    n_grid = int(cfg["n_grid"])
    n_design = int(cfg["n_design"])

    f0 = 3.0e9
    lam0 = C0 / f0
    lx = 2.4 * lam0
    ly = 1.6 * lam0
    dz = lx / n_grid
    w_in = 0.28 * lam0
    w_out = 0.56 * lam0
    d_box = 0.8 * lam0
    eps_bg = 2.1
    eps_wg = 6.0

    def make_solver():
        grid = wakis.GridFIT3D(
            -0.5 * lx, 0.5 * lx, -0.5 * ly, 0.5 * ly, -0.5 * dz, 0.5 * dz, n_grid, n_grid, 1, verbose=0
        )
        solver = wakis.SolverFIT3D(
            grid,
            bc_low=["cpml", "cpml", "periodic"],
            bc_high=["cpml", "cpml", "periodic"],
            n_pml=6,
            verbose=0,
        )
        bg = MaterialGrid((0.0, 0.0, 0.0), (lx, ly, 0.0), (1, 1, 1), eps_min=eps_bg, eps_max=eps_bg)
        bg.apply(solver, np.ones((1, 1, 1)))
        in_wg = MaterialGrid((-0.35 * lx, 0.0, 0.0), (0.5 * lx, w_in, 0.0), (1, 1, 1), eps_min=eps_wg, eps_max=eps_wg)
        in_wg.apply(solver, np.ones((1, 1, 1)))
        out_wg = MaterialGrid((0.35 * lx, 0.0, 0.0), (0.5 * lx, w_out, 0.0), (1, 1, 1), eps_min=eps_wg, eps_max=eps_wg)
        out_wg.apply(solver, np.ones((1, 1, 1)))
        return solver

    design = MaterialGrid(
        (0.0, 0.0, 0.0),
        (d_box, d_box, 0.0),
        (n_design, n_design, 1),
        eps_min=eps_bg,
        eps_max=eps_wg,
        smoothing="tensor",
    )
    src = CurrentSource(
        GaussianSource(f0, 0.5 * f0),
        component="z",
        center=(-0.32 * lx, 0.0, 0.0),
        size=(0.0, w_in, 0.0),
    )
    # Two monitors on the upper and lower lobes of the output dual-mode guide to measure antisymmetric TE1-like excitation
    monitors = {
        "upper": wakis.DFTMonitor([f0], components=["Ez"], center=(0.30 * lx, 0.16 * lam0, 0.0), size=(0.0, 0.14 * lam0, 0.0)),
        "lower": wakis.DFTMonitor([f0], components=["Ez"], center=(0.30 * lx, -0.16 * lam0, 0.0), size=(0.0, 0.14 * lam0, 0.0)),
    }
    sim = AdjointSimulation(make_solver, [src], monitors, design, n_steps)

    # Initialize with an antisymmetric density ramp so the gradient drives antisymmetric TE0 -> TE1 conversion
    yy_mg = np.linspace(-1.0, 1.0, n_design)
    rho = np.clip(0.5 + 0.15 * np.outer(np.ones(n_design), yy_mg)[:, :, None], 0.05, 0.95)

    history_j = []
    history_gnorm = []
    step_size = 0.18

    for _it in range(n_iters):
        out_dft = sim.forward(rho)
        ez_u = out_dft["upper"]["Ez"]
        ez_l = out_dft["lower"]["Ez"]
        diff = ez_u - ez_l
        obj_val = float(np.sum(np.abs(diff) ** 2))

        g = sim.vjp({"upper": {"Ez": 2.0 * diff}, "lower": {"Ez": -2.0 * diff}})
        gnorm = float(np.linalg.norm(g))
        history_j.append(obj_val)
        history_gnorm.append(gnorm)
        if gnorm > 0:
            rho = np.clip(rho + step_size * g / (float(np.max(np.abs(g))) + 1e-30), 0.0, 1.0)

    out = {
        "label": "PEDAGOGICAL_TOY_DEMO_NOT_INVRS_GYM",
        "mode": mode,
        "n_iters": n_iters,
        "n_steps": n_steps,
        "objective_history": history_j,
        "grad_norm_history": history_gnorm,
        "objective_ratio_end_over_start": float(history_j[-1] / max(history_j[0], 1e-30)),
    }
    (DATA_DIR / "toy_optimization_summary.json").write_text(json.dumps(out, indent=2))
    print(
        f"[Task 3] Toy adjoint demo ({n_iters} iters): J_start={history_j[0]:.4e} -> "
        f"J_end={history_j[-1]:.4e} ({out['objective_ratio_end_over_start']:.2f}x)"
    )
    return out


def run_task_4_optional_invrs_gym() -> dict:
    """Task 4: Optional official invrs-gym re-score if invrs_gym and ceviche_challenges are installed."""
    try:
        import invrs_gym  # type: ignore[import-not-found]
    except ImportError:
        msg = (
            "Optional package `invrs_gym` is not installed in this offline environment. "
            "Using pre-computed official scores from `data/round2_*/summary.json` and `leaderboard_rescore.json`."
        )
        print(f"[Task 4] {msg}")
        return {"installed": False, "message": msg}

    return {"installed": True, "version": getattr(invrs_gym, "__version__", "unknown")}


def plot_all() -> list[Path]:
    """Generate all 8 tutorial figures and the design-evolution GIF in ``figures/``."""
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    sp_summary = json.loads((DATA_DIR / "round2_splitter" / "summary.json").read_text())
    mc_summary = json.loads((DATA_DIR / "round2_mode_converter" / "summary.json").read_text())
    sp_fc = np.load(DATA_DIR / "round2_splitter" / "fields_compact.npz")
    mc_fc = np.load(DATA_DIR / "round2_mode_converter" / "fields_compact.npz")
    sp_snaps = np.load(DATA_DIR / "round2_splitter" / "snapshots_thinned.npz")
    mc_snaps = np.load(DATA_DIR / "round2_mode_converter" / "snapshots_thinned.npz")
    sp_design = np.load(DATA_DIR / "round2_splitter" / "design.npy")
    if sp_design.ndim == 3:
        sp_design = sp_design[:, :, 0]
    mc_design = np.load(DATA_DIR / "round2_mode_converter" / "design.npy")
    if mc_design.ndim == 3:
        mc_design = mc_design[:, :, 0]
    resim = np.load(DATA_DIR / "wakis_resim_full.npz")
    lb_data = json.loads(LEADERBOARD_PATH.read_text())

    saved: list[Path] = []
    eval_wl_nm = np.array([1265.0, 1270.0, 1275.0, 1285.0, 1290.0, 1295.0])

    # --- Figure 1: Final binary designs + morphological length-scale verification ---
    fig, axes = plt.subplots(2, 2, figsize=(11, 8.8), constrained_layout=True)
    brush8 = circular_brush(8)
    cross = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)
    for row, (title, des, summ) in enumerate(
        [
            ("ceviche_power_splitter", sp_design, sp_summary),
            ("ceviche_mode_converter", mc_design, mc_summary),
        ]
    ):
        ax0, ax1 = axes[row, 0], axes[row, 1]
        b = des >= 0.5
        im0 = ax0.imshow(
            des.T,
            origin="lower",
            cmap="gray_r",
            vmin=0.0,
            vmax=1.0,
            extent=[0, 1.6, 0, 1.6],
            aspect="equal",
        )
        ls_px = summ["length_scale_leaderboard_px"]
        ax0.set_title(
            f"{title}: Final Design (160x160 px, 10 nm/px)\n"
            f"Official eval_metric = {summ['official_eval_metric']:+.4f}, LS = {ls_px[0]}/{ls_px[1]} px (80 nm)"
        )
        ax0.set_xlabel("x [um]")
        ax0.set_ylabel("y [um]")
        fig.colorbar(im0, ax=ax0, fraction=0.046, pad=0.04, label="Density rho")

        # Morphological opening + closing residual map with 8 px circular brush on interior pixels
        b_pad = np.pad(b, 10, mode="edge")
        opened = binary_opening(b_pad, structure=brush8)[10:-10, 10:-10]
        closed = binary_closing(b_pad, structure=brush8)[10:-10, 10:-10]
        int_solid = binary_erosion(b, structure=cross, border_value=1)
        int_void = binary_erosion(~b, structure=cross, border_value=1)
        n_viol = int(np.sum(((b != opened) & int_solid) | ((b != closed) & int_void)))

        ax1.imshow(
            des.T,
            origin="lower",
            cmap="Greys",
            vmin=0.0,
            vmax=1.3,
            extent=[0, 1.6, 0, 1.6],
            aspect="equal",
            alpha=0.35,
        )
        ax1.contour(
            np.linspace(0, 1.6, des.shape[0]),
            np.linspace(0, 1.6, des.shape[1]),
            des.T,
            levels=[0.5],
            colors=["#1f77b4"],
            linewidths=1.4,
        )
        # Draw 80 nm (8 px) reference ruler circle in the corner
        theta = np.linspace(0, 2 * np.pi, 100)
        ax1.plot(0.18 + 0.04 * np.cos(theta), 0.18 + 0.04 * np.sin(theta), "r-", lw=2, label="80 nm (8 px) brush")
        ax1.set_title(
            f"8 px (80 nm) Morphological Check: min_solid={ls_px[0]} px, min_void={ls_px[1]} px\n"
            f"(Interior opening/closing violations at 8 px = {n_viol} px)"
        )
        ax1.set_xlabel("x [um]")
        ax1.set_ylabel("y [um]")
        ax1.legend(loc="upper right", fontsize=8)

    p1 = FIG_DIR / "01_final_designs_and_length_scale.png"
    fig.savefig(p1, dpi=150)
    plt.close(fig)
    saved.append(p1)

    # --- Figure 2: Optimization traces with Two-Stage beta continuation ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex="col", constrained_layout=True)
    for col, (name, snaps, summ) in enumerate(
        [
            ("Power Splitter (seed 1)", sp_snaps, sp_summary),
            ("Mode Converter (seed 1)", mc_snaps, mc_summary),
        ]
    ):
        iters = snaps["kept_index"].astype(int)
        losses = snaps["loss"].astype(float)
        evals = snaps["eval_metric"].astype(float)
        stages = snaps["stage"].astype(str)
        betas = np.array([float(st.split("beta")[-1]) for st in stages], dtype=float)
        ste_mask = stages == "ste_beta64"
        ste_start_iter = int(iters[ste_mask][0]) if np.any(ste_mask) else 120

        ax_top = axes[0, col]
        ax_bot = axes[1, col]

        ax_top.plot(iters, losses, color="#1f77b4", lw=1.8, label="Scalar softplus loss L")
        ax_top.axvline(ste_start_iter, color="#d62728", ls="--", lw=1.4, label=f"Stage 2 STE repair (step {ste_start_iter})")
        ax_top.set_ylabel("Surrogate Loss L")
        ax_top.set_title(f"{name}: Loss & beta Continuation")
        ax_top.grid(True, alpha=0.3)
        ax_top_b = ax_top.twinx()
        ax_top_b.step(iters, betas, where="post", color="#ff7f0e", alpha=0.65, lw=1.3, label="Projection beta")
        ax_top_b.set_ylabel("Projection sharpness beta", color="#ff7f0e")
        ax_top.legend(loc="upper right", fontsize=8)

        ax_bot.plot(iters, evals, color="#2ca02c", lw=1.8, label="Surrogate eval_metric (r=1,2)")
        ax_bot.axhline(
            summ["official_eval_metric"],
            color="#9467bd",
            ls="-.",
            lw=1.6,
            label=f"Official FDFD eval = {summ['official_eval_metric']:+.4f}",
        )
        ax_bot.axhline(0.0, color="k", ls=":", lw=1.0, alpha=0.6)
        ax_bot.axvline(ste_start_iter, color="#d62728", ls="--", lw=1.4)
        # Mark STE candidate checkpoints
        for cand in summ["candidates"]:
            c_tag = f"{cand['stage']}:{cand['step']}"
            c_it = ste_start_iter + int(cand["step"])
            ax_bot.scatter(
                [c_it],
                [cand["official_eval_metric"]],
                color="#d62728" if c_tag == summ.get("selected_stage") else "#8c564b",
                s=55,
                zorder=5,
                label=f"{c_tag} ({cand['official_eval_metric']:+.3f})",
            )
        ax_bot.set_xlabel("Optimization Step")
        ax_bot.set_ylabel("Orthotope eval_metric")
        ax_bot.set_title(f"{name}: Orthotope Metric & Candidates")
        ax_bot.grid(True, alpha=0.3)
        ax_bot.legend(loc="lower right", fontsize=7.5)

    p2 = FIG_DIR / "02_optimization_traces_and_beta_schedule.png"
    fig.savefig(p2, dpi=150)
    plt.close(fig)
    saved.append(p2)

    # --- Figure 3: Power Splitter spectra (Wakis re-simulation vs stored modal, estimator, and official FDFD) ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    dense_wl_nm = resim["dense_wavelengths_nm"]
    sp_dense_resim = resim["splitter_s2_dense"]
    sp_dense_S = (sp_fc["dense_S_re"].astype(np.float64) + 1j * sp_fc["dense_S_im"].astype(np.float64)) * float(sp_fc["dense_S_scale"])
    sp_dense_stored = np.abs(sp_dense_S) ** 2
    sp_off_s2 = np.asarray(sp_summary["official_S2"])[:, 0, :]
    sp_mod_s2 = np.asarray(sp_summary["wakis_r1_modal_S2"])[:, 0, :]
    sp_est_s2 = np.asarray(sp_summary["wakis_r1_ceviche_estimator_S2"])[:, 0, :]

    ax = axes[0]
    ax.plot(dense_wl_nm, sp_dense_stored[:, 1], "-", color="#1f77b4", lw=2.0, label="Stored Wakis |S21|^2 (Port 1)")
    ax.plot(dense_wl_nm, sp_dense_resim[:, 1], "--", color="#ff7f0e", lw=1.8, label="Re-simulated Wakis |S21|^2")
    ax.plot(dense_wl_nm, sp_dense_stored[:, 2], "-", color="#2ca02c", lw=2.0, label="Stored Wakis |S31|^2 (Port 2)")
    ax.plot(dense_wl_nm, sp_dense_resim[:, 2], ":", color="#d62728", lw=1.8, label="Re-simulated Wakis |S31|^2")
    ax.scatter(eval_wl_nm, sp_off_s2[:, 1], marker="o", s=50, color="#1f77b4", edgecolor="k", zorder=5, label="Official FDFD |S21|^2")
    ax.scatter(eval_wl_nm, sp_off_s2[:, 2], marker="s", s=50, color="#2ca02c", edgecolor="k", zorder=5, label="Official FDFD |S31|^2")
    ax.scatter(eval_wl_nm, sp_est_s2[:, 1], marker="^", s=45, color="#9467bd", zorder=5, label="Wakis ceviche-est |S21|^2")
    ax.fill_between([1265, 1295], SPLITTER_LB_F32, 1.0, color="#2ca02c", alpha=0.12, label="Target window [0.4467, 1.0]")
    ax.set_xlim(1245, 1315)
    ax.set_ylim(0.38, 0.56)
    ax.set_xlabel("Wavelength [nm]")
    ax.set_ylabel("Power Transmission |S_j1|^2")
    ax.set_title("Power Splitter: Transmission Arms (Ports 1 & 2)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower left", fontsize=7.5, ncol=2)

    ax = axes[1]
    ax.plot(dense_wl_nm, 10 * np.log10(np.maximum(sp_dense_stored[:, 0], 1e-7)), "-", color="#1f77b4", lw=2.0, label="Stored Wakis |S11|^2 [dB]")
    ax.plot(dense_wl_nm, 10 * np.log10(np.maximum(sp_dense_resim[:, 0], 1e-7)), "--", color="#ff7f0e", lw=1.8, label="Re-simulated Wakis |S11|^2 [dB]")
    ax.scatter(eval_wl_nm, 10 * np.log10(np.maximum(sp_off_s2[:, 0], 1e-7)), marker="o", s=55, color="#d62728", edgecolor="k", zorder=5, label="Official FDFD |S11|^2 [dB]")
    ax.scatter(eval_wl_nm, 10 * np.log10(np.maximum(sp_mod_s2[:, 0], 1e-7)), marker="d", s=45, color="#2ca02c", zorder=5, label="Wakis modal |S11|^2 [dB]")
    ax.axhline(-20.0, color="#d62728", ls="--", lw=1.4, label="Reflection upper bound (-20 dB)")
    ax.fill_between([1265, 1295], -45, -20, color="#2ca02c", alpha=0.10, label="Target window (<= -20 dB)")
    ax.set_xlim(1245, 1315)
    ax.set_ylim(-42, -14)
    ax.set_xlabel("Wavelength [nm]")
    ax.set_ylabel("Return Loss 10 log10(|S11|^2) [dB]")
    ax.set_title("Power Splitter: Input Reflection (Port 0)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", fontsize=8)

    p3 = FIG_DIR / "03_splitter_spectra_comparison.png"
    fig.savefig(p3, dpi=150)
    plt.close(fig)
    saved.append(p3)

    # --- Figure 4: Mode Converter spectra & loss budget (why it falls short of leaderboard) ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    mc_dense_resim = resim["modeconv_s2_dense"]
    mc_dense_S = (mc_fc["dense_S_re"].astype(np.float64) + 1j * mc_fc["dense_S_im"].astype(np.float64)) * float(mc_fc["dense_S_scale"])
    mc_dense_stored = np.abs(mc_dense_S) ** 2
    mc_off_s2 = np.asarray(mc_summary["official_S2"])[:, 0, :]
    mc_est_s2 = np.asarray(mc_summary["wakis_r1_ceviche_estimator_S2"])[:, 0, :]

    ax = axes[0]
    ax.plot(dense_wl_nm, mc_dense_stored[:, 1], "-", color="#1f77b4", lw=2.0, label="Stored Wakis TE0->TE1 |S21|^2")
    ax.plot(dense_wl_nm, mc_dense_resim[:, 1], "--", color="#ff7f0e", lw=1.8, label="Re-simulated Wakis |S21|^2")
    ax.scatter(eval_wl_nm, mc_off_s2[:, 1], marker="o", s=55, color="#2ca02c", edgecolor="k", zorder=5, label="Official FDFD |S21|^2")
    ax.scatter(eval_wl_nm, mc_est_s2[:, 1], marker="^", s=45, color="#9467bd", zorder=5, label="Wakis ceviche-est |S21|^2")
    ax.axhline(MODECONV_LB_F32, color="#d62728", ls="--", lw=1.4, label="Target lower bound (-0.5 dB = 0.8913)")
    ax.axhline(0.89125 + 0.8536 * (1.0 - 0.89125), color="0.4", ls=":", lw=1.3, label="Leaderboard 8/8 px level (~0.984)")
    ax.fill_between([1265, 1295], MODECONV_LB_F32, 1.0, color="#2ca02c", alpha=0.12, label="Target window [0.8913, 1.0]")
    ax.set_xlim(1245, 1315)
    ax.set_ylim(0.80, 1.01)
    ax.set_xlabel("Wavelength [nm]")
    ax.set_ylabel("TE0 -> TE1 Conversion |S21|^2")
    ax.set_title("Mode Converter: Conversion Spectrum vs Target Window")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower left", fontsize=7.5)

    ax = axes[1]
    x_idx = np.arange(len(eval_wl_nm))
    w_bar = 0.26
    refl_pct = 100.0 * mc_off_s2[:, 0]
    unacc_pct = 100.0 * (1.0 - mc_off_s2[:, 0] - mc_off_s2[:, 1])
    shortfall_to_unity_pct = 100.0 * (1.0 - mc_off_s2[:, 1])
    ax.bar(x_idx - w_bar, refl_pct, width=w_bar, color="#1f77b4", label="Back-reflection |S11|^2 (< 0.16%)")
    ax.bar(x_idx, unacc_pct, width=w_bar, color="#ff7f0e", label="Radiation + unconverted TE0 (4.5% - 7.4%)")
    ax.bar(x_idx + w_bar, shortfall_to_unity_pct, width=w_bar, color="#d62728", label="Total TE1 shortfall 1 - |S21|^2")
    ax.axhline(100.0 * (1.0 - MODECONV_LB_F32), color="k", ls="--", lw=1.2, label="Max allowed loss for eval >= 0 (10.87%)")
    ax.set_xticks(x_idx)
    ax.set_xticklabels([f"{int(w)} nm" for w in eval_wl_nm])
    ax.set_ylabel("Power Fraction [%]")
    ax.set_title("Mode Converter Loss Budget: Radiation/TE0 Dominates Reflection")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(loc="upper right", fontsize=8)

    p4 = FIG_DIR / "04_mode_converter_spectra_and_gap_analysis.png"
    fig.savefig(p4, dpi=150)
    plt.close(fig)
    saved.append(p4)

    # --- Figure 5: Splitter DFT fields at all 6 challenge wavelengths ---
    fig, axes = plt.subplots(2, 3, figsize=(13.5, 6.8), sharex=True, sharey=True, constrained_layout=True)
    sp_ez = (sp_fc["Ez_re"].astype(np.float64) + 1j * sp_fc["Ez_im"].astype(np.float64)) * float(sp_fc["Ez_scale"])
    sp_eps = sp_fc["eps"].astype(np.float64) * float(sp_fc["eps_scale"])
    lx_um, ly_um = float(sp_fc["x_um"][-1]), float(sp_fc["y_um"][-1])
    vmax_sp = float(np.percentile(np.abs(sp_ez), 99.2))
    for k, wl in enumerate(eval_wl_nm):
        ax = axes[k // 3, k % 3]
        ez_real = np.real(sp_ez[k]).T
        im = ax.imshow(
            ez_real,
            origin="lower",
            cmap="RdBu_r",
            vmin=-vmax_sp,
            vmax=vmax_sp,
            extent=[0, lx_um, 0, ly_um],
            aspect="equal",
        )
        ax.contour(
            np.linspace(0, lx_um, sp_eps.shape[0]),
            np.linspace(0, ly_um, sp_eps.shape[1]),
            sp_eps.T,
            levels=[7.0],
            colors=["k"],
            linewidths=0.65,
            alpha=0.7,
        )
        s21 = sp_off_s2[k, 1]
        s31 = sp_off_s2[k, 2]
        ax.set_title(f"lambda = {int(wl)} nm: |S21|^2={s21:.3f}, |S31|^2={s31:.3f}", fontsize=9.5)
        if k // 3 == 1:
            ax.set_xlabel("x [um]")
        if k % 3 == 0:
            ax.set_ylabel("y [um]")
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="Re(E_z) [arb. units]")
    fig.suptitle("Power Splitter: Steady-State DFT Field Re(E_z) Across All 6 Challenge Wavelengths", fontsize=12)

    p5 = FIG_DIR / "05_splitter_fields_wavelengths.png"
    fig.savefig(p5, dpi=150)
    plt.close(fig)
    saved.append(p5)

    # --- Figure 6: Mode Converter DFT fields at all 6 challenge wavelengths ---
    fig, axes = plt.subplots(2, 3, figsize=(13.5, 6.8), sharex=True, sharey=True, constrained_layout=True)
    mc_ez = (mc_fc["Ez_re"].astype(np.float64) + 1j * mc_fc["Ez_im"].astype(np.float64)) * float(mc_fc["Ez_scale"])
    mc_eps = mc_fc["eps"].astype(np.float64) * float(mc_fc["eps_scale"])
    vmax_mc = float(np.percentile(np.abs(mc_ez), 99.2))
    for k, wl in enumerate(eval_wl_nm):
        ax = axes[k // 3, k % 3]
        ez_real = np.real(mc_ez[k]).T
        im = ax.imshow(
            ez_real,
            origin="lower",
            cmap="RdBu_r",
            vmin=-vmax_mc,
            vmax=vmax_mc,
            extent=[0, lx_um, 0, ly_um],
            aspect="equal",
        )
        ax.contour(
            np.linspace(0, lx_um, mc_eps.shape[0]),
            np.linspace(0, ly_um, mc_eps.shape[1]),
            mc_eps.T,
            levels=[7.0],
            colors=["k"],
            linewidths=0.65,
            alpha=0.7,
        )
        s21 = mc_off_s2[k, 1]
        s11 = mc_off_s2[k, 0]
        ax.set_title(f"lambda = {int(wl)} nm: TE1 |S21|^2={s21:.3f}, |S11|^2={s11:.4f}", fontsize=9.5)
        if k // 3 == 1:
            ax.set_xlabel("x [um]")
        if k % 3 == 0:
            ax.set_ylabel("y [um]")
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="Re(E_z) [arb. units]")
    fig.suptitle("Mode Converter: Steady-State DFT Field Re(E_z) (TE0 Input -> Antisymmetric TE1 Output)", fontsize=12)

    p6 = FIG_DIR / "06_mode_converter_fields_wavelengths.png"
    fig.savefig(p6, dpi=150)
    plt.close(fig)
    saved.append(p6)

    # --- Figure 7: Leaderboard & Round-1 vs Round-2 Comparison ---
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.0), constrained_layout=True)
    lb_sp_best_entry = next(
        e for e in lb_data
        if e["challenge"] == "ceviche_power_splitter" and not str(e["file"]).startswith("ours_")
    )
    lb_sp_ours_rescore = next(
        e for e in lb_data
        if e["challenge"] == "ceviche_power_splitter" and str(e["file"]).startswith("ours_")
    )

    ax = axes[0]
    labels_sp = [
        f"LB Best ({lb_sp_best_entry['minimum_width']}/{lb_sp_best_entry['minimum_spacing']} px, 80 nm)",
        "Ours Round 2 Pilot (3/3-4/4 px, 40 nm)",
        "Ours Round 1 Seed 1 (1/2 px, infeasible)",
        "Ours Round 1 Seed 0 (1/2 px, infeasible)",
        f"Ours Round 2 Rescore ({lb_sp_ours_rescore['minimum_width']}/{lb_sp_ours_rescore['minimum_spacing']} px, 80 nm)",
        "Ours Round 2 Official (8/8 px, 80 nm)",
    ]
    vals_sp = [
        float(lb_sp_best_entry["eval_metric"]),
        0.0520,
        0.057933,
        0.066941,
        float(lb_sp_ours_rescore["eval_metric"]),
        float(sp_summary["official_eval_metric"]),
    ]
    colors_sp = ["#7f7f7f", "#9467bd", "#ff7f0e", "#ff7f0e", "#1f77b4", "#2ca02c"]
    y_pos = np.arange(len(labels_sp))
    bars = ax.barh(y_pos, vals_sp, color=colors_sp, edgecolor="k", height=0.62)
    ax.axvline(float(lb_sp_best_entry["eval_metric"]), color="#7f7f7f", ls="--", lw=1.3, label="Leaderboard best (+0.0090 at 9/8 px)")
    ax.axvline(SPLITTER_PASSIVITY_CEILING, color="#d62728", ls=":", lw=1.3, label="Passivity ceiling (+0.0964)")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels_sp, fontsize=8.5)
    ax.set_xlabel("Official Orthotope eval_metric (higher is better)")
    ax.set_title("ceviche_power_splitter: Leaderboard vs Ours")
    ax.grid(True, axis="x", alpha=0.3)
    for bar, v in zip(bars, vals_sp):
        ax.text(max(v + 0.0015, 0.002), bar.get_y() + 0.5 * bar.get_height(), f"{v:+.4f}", va="center", fontsize=8)
    ax.set_xlim(-0.005, 0.105)
    ax.legend(loc="lower right", fontsize=8)

    ax = axes[1]
    labels_mc = [
        "Ours Round 1 Seed 0 (2/3 px, infeasible)",
        "Ours Round 2 Official (8/8 px, 80 nm)",
        "Ours Round 2 Pilot (3/3 px, 40 nm)",
        "Ours Stage-1 Peak Surrogate (unrepaired)",
        "Leaderboard Best Feasible (8/8 px, 80 nm)",
        "Leaderboard Overall Best (6/6 px, 60 nm)",
    ]
    vals_mc = [
        -0.310085,
        float(mc_summary["official_eval_metric"]),
        0.3810,
        float(np.max(mc_snaps["eval_metric"][mc_snaps["stage"] != "ste_beta64"])),
        0.8536,
        0.9095,
    ]
    colors_mc = ["#ff7f0e", "#d62728", "#9467bd", "#8c564b", "#1f77b4", "#1f77b4"]
    y_pos_mc = np.arange(len(labels_mc))
    bars_mc = ax.barh(y_pos_mc, vals_mc, color=colors_mc, edgecolor="k", height=0.58)
    ax.axvline(0.0, color="k", ls=":", lw=1.0)
    ax.set_yticks(y_pos_mc)
    ax.set_yticklabels(labels_mc, fontsize=8.5)
    ax.set_xlabel("Orthotope eval_metric (higher is better)")
    ax.set_title("ceviche_mode_converter: Honest Leaderboard Comparison")
    ax.grid(True, axis="x", alpha=0.3)
    for bar, v in zip(bars_mc, vals_mc):
        x_txt = v + 0.02 if v >= 0 else v + 0.02
        ax.text(max(x_txt, 0.02), bar.get_y() + 0.5 * bar.get_height(), f"{v:+.4f}", va="center", fontsize=8)
    ax.set_xlim(-0.42, 1.08)

    p7 = FIG_DIR / "07_leaderboard_and_round_comparison.png"
    fig.savefig(p7, dpi=150)
    plt.close(fig)
    saved.append(p7)

    # --- Figure 8: Design-evolution animated GIF (< 5 MB) from snapshots_thinned.npz ---
    sp_dens = sp_snaps["designs"].astype(np.float32)
    mc_dens = mc_snaps["designs"].astype(np.float32)
    sp_it = sp_snaps["kept_index"].astype(int)
    sp_ev = sp_snaps["eval_metric"].astype(float)
    mc_ev = mc_snaps["eval_metric"].astype(float)
    sp_stages = sp_snaps["stage"].astype(str)

    n_frames = min(len(sp_it), len(mc_dens))
    frame_indices = np.linspace(0, n_frames - 1, min(18, n_frames), dtype=int)

    fig_gif, axes_gif = plt.subplots(1, 2, figsize=(8.5, 4.0), constrained_layout=True)

    def _update_frame(idx_pos: int):
        f_idx = int(frame_indices[idx_pos])
        for ax in axes_gif:
            ax.clear()
        it_num = int(sp_it[f_idx])
        st_str = str(sp_stages[f_idx])
        beta_val = float(st_str.split("beta")[-1])
        stage_lbl = "Stage 1 (r=1+r=2)" if st_str != "ste_beta64" else "Stage 2 (80 nm Repair)"

        axes_gif[0].imshow(
            sp_dens[f_idx].T,
            origin="lower",
            cmap="gray_r",
            vmin=0.0,
            vmax=1.0,
            extent=[0, 1.6, 0, 1.6],
            aspect="equal",
        )
        axes_gif[0].set_title(
            f"Power Splitter | step {it_num} ({stage_lbl})\n"
            f"beta={beta_val:.0f}, surrogate eval={sp_ev[f_idx]:+.3f}",
            fontsize=9,
        )
        axes_gif[0].set_xlabel("x [um]")
        axes_gif[0].set_ylabel("y [um]")

        axes_gif[1].imshow(
            mc_dens[f_idx].T,
            origin="lower",
            cmap="gray_r",
            vmin=0.0,
            vmax=1.0,
            extent=[0, 1.6, 0, 1.6],
            aspect="equal",
        )
        axes_gif[1].set_title(
            f"Mode Converter | step {it_num} ({stage_lbl})\n"
            f"beta={beta_val:.0f}, surrogate eval={mc_ev[f_idx]:+.3f}",
            fontsize=9,
        )
        axes_gif[1].set_xlabel("x [um]")
        axes_gif[1].set_ylabel("y [um]")
        return axes_gif

    anim = animation.FuncAnimation(fig_gif, _update_frame, frames=len(frame_indices), interval=350)
    p8 = FIG_DIR / "08_design_evolution.gif"
    anim.save(p8, writer=animation.PillowWriter(fps=3), dpi=90)
    plt.close(fig_gif)
    saved.append(p8)

    print(f"[Plot] Generated {len(saved)} figures in {FIG_DIR}")
    return saved

## Multi-Resolution Objective, Tensor Smoothing & SSP Pilot (`studies/ssp`)

During Round-2 development, two mechanisms were evaluated to prevent narrow resonant features from
exploiting coarse-grid dispersion between the $40\,\text{nm}$ ($r=1$) / $20\,\text{nm}$ ($r=2$)
optimization grids and the $10\,\text{nm}$ official `ceviche` FDFD evaluator:

1. **Robust multi-resolution objective (`multi_resolution_loss`)**: averaging the scalar loss
   and gradients across $r=1$ ($40\,\text{nm}$) and $r=2$ ($20\,\text{nm}$) Wakis grids so that
   a candidate must satisfy the target orthotope at both grid spacings simultaneously.
2. **Anisotropic tensor subpixel smoothing (`smoothing="tensor"`) and Subpixel-Smoothed Projection
   (SSP; Hammond et al., *Opt. Express* **33**, 33620, 2025, arXiv:`2503.20189`)**: the pilot
   study in `studies/ssp/data/pilot.md` compared baseline scalar projection against SSP on the
   $40\,\text{nm}$ optimization grid (4 seeds on `waveguide_bend`, 2 seeds on `power_splitter`;
   **indications only** due to small seed counts):

| Challenge | Arm | Seeds | Optimize Grid ($40\,\text{nm}$) Mean `eval` | Holdout Grid ($20\,\text{nm}$) Mean `eval` | Best Holdout `eval` | Optimize $\to$ Holdout Drop |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| `ceviche_lightweight_waveguide_bend` | Baseline | 4 | $+0.5060$ | $+0.0566$ | $+0.5177$ | $-0.4494$ |
| `ceviche_lightweight_waveguide_bend` | SSP | 4 | $+0.2442$ | $+0.0725$ | $+0.5256$ | **$-0.1717$** |
| `ceviche_lightweight_power_splitter` | Baseline | 2 | $+0.0706$ | $+0.0176$ | $+0.0680$ | $-0.0531$ |
| `ceviche_lightweight_power_splitter` | SSP | 2 | $+0.0493$ | $-0.1636$ | $+0.0462$ | $-0.2129$ |

Because 2–4 seeds per arm have large run-to-run variance, these pilot numbers are treated as
exploratory indications only; for full-resolution Round-2 runs (`10 nm` design grid),
`PaddedMaterialGrid(..., smoothing="tensor")` combined with `multi_resolution_loss` ($r=1, 2$)
and multi-candidate STE checkpoint selection (`candidates` in `summary.json`) provided the
strongest verified official scores (`+0.0682` on `power_splitter`, `+0.3122` on `mode_converter`).

## Limits & Failure Modes

1. **Single-seed / two-seed stochastic sensitivity (`ceviche_mode_converter` gap to leaderboard)**:
   On `ceviche_mode_converter`, our Round-2 production run (`seed 1`) reaches `+0.3122` at `8/8 px` (`80 nm`),
   well below the `invrs-gym` leaderboard (`+0.8536` at `8/8 px`, `+0.9095` at `6/6 px`). Inspection of the optimization
   trace shows that Stage 1 reached a surrogate `eval_metric` of `+0.4566` at step 102 (`density_beta32`), but the hard
   morphological projection step at the start of Stage 2 (`ste_beta64:0`) dropped the metric to `+0.2598` (`+0.2623` on
   official FDFD) by rounding sub-80 nm phase-matching trenches, before recovering to `+0.3082` (`+0.3122` on official FDFD)
   at `ste_beta64:2`. Across all 6 wavelengths, back-reflection is negligible ($\le 0.059\% = -32.3\,\text{dB}$ vs. the
   $1\%$ ceiling), whereas $6.60\%\text{--}7.48\%$ of the input power is lost to cladding radiation or unconverted
   $\mathrm{TE}_0$ power, with the worst bottleneck at $\lambda_1 = 1265\,\text{nm}$ ($|S_{21}|^2 = 0.9252$).
   Lookahead morphological penalties during Stage 1, cosine-decayed step sizes in Stage 2, and a 4–8 seed portfolio are
   concrete, testable hypotheses to close this gap.
2. **2D effective-index vs. 3D SOI radiation loss**:
   Both challenges use the 2D `ceviche` model ($\varepsilon_{\min} = 2.085$, $\varepsilon_{\max} = 12.110$). Out-of-plane
   scattering into the $\mathrm{SiO}_2$ cladding of a 3D $220\,\text{nm}$ SOI slab is not captured by a 2D simulation.
3. **STE boundary oscillation in Stage 2**:
   Because the morphological repair operator (`imageruler` opening/closing) is piecewise constant, a fixed Adam step size
   in `ste_beta64` can flip boundary pixels back and forth (for example, `mode_converter` oscillated between `+0.3082`
   at `ste_beta64:2`, `+0.2528` at `ste_beta64:5`, and `+0.3049` at `ste_beta64:6`). Evaluating the top-4 STE checkpoints
   on the $r=1$ ($10\,\text{nm}$) grid and selecting `ste_beta64:2` (`+0.3122`) and `ste_beta64:11` (`+0.0682`) mitigates
   final-step oscillation.

## Verified References

1. M. F. Schubert et al., "invrs-gym: a toolkit for nanophotonic inverse design research," arXiv:`2410.24132` (2024).
2. M. F. Schubert, A. K. Cheung, I. A. D. Williamson, A. Spyra, and D. H. Alexander, "Inverse design of photonic devices
   with strict foundry fabrication constraints," *ACS Photonics* **9**, 2327–2336 (2022), DOI: `10.1021/acsphotonics.2c00313`.
3. T. W. Hughes, I. A. D. Williamson, M. Minkov, and S. Fan, "Forward-mode differentiation of Maxwell's equations,"
   *ACS Photonics* **6**, 3010–3016 (2019), arXiv:`1908.10507`, DOI: `10.1021/acsphotonics.9b01238`.
4. F. Wang, B. S. Lazarov, and O. Sigmund, "On projection methods, convergence and robust formulations in topology
   optimization," *Struct. Multidiscip. Optim.* **43**, 767–784 (2011), DOI: `10.1007/s00158-010-0602-y`.
5. M. Chen, R. E. Christiansen, J. A. Fan, G. Işiklar, J. Jiang, S. G. Johnson et al., "Validation and characterization of
   algorithms and software for photonics inverse design," *J. Opt. Soc. Am. B* **41**, A161–A176 (2024),
   DOI: `10.1364/JOSAB.506412`.
6. A. M. Hammond, A. Oskooi, I. A. D. Williamson, and S. G. Johnson, "Subpixel-smoothed projection for topology
   optimization," *Opt. Express* **33**, 33620 (2025), arXiv:`2503.20189`.

In [ ]:
def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    mode_group = parser.add_mutually_exclusive_group()
    mode_group.add_argument("--quick", action="store_true", help="Fast verification mode (< 1 min).")
    mode_group.add_argument("--full", action="store_true", help="Full native-resolution re-simulation mode.")
    parser.add_argument(
        "--task",
        choices=list(TASKS.keys()),
        default=None,
        help="Run only the specified task.",
    )
    parser.add_argument("--list-tasks", action="store_true", help="Print available tasks and exit.")
    parser.add_argument("--plot", action="store_true", help="Regenerate figures from stored data only.")
    args = parser.parse_args(argv)

    if args.list_tasks:
        for name, desc in TASKS.items():
            print(f"{name}: {desc}")
        return 0

    mode = "full" if args.full else "quick"

    if args.plot and args.task is None:
        plot_all()
        return 0

    selected = [args.task] if args.task else ["metrics_and_length_scale", "wakis_resimulation", "gap_and_candidate_analysis", "toy_optimization", "optional_invrs_gym"]

    if "metrics_and_length_scale" in selected:
        run_task_0_metrics_and_length_scale()
    if "wakis_resimulation" in selected:
        run_task_1_wakis_resimulation(mode=mode)
    if "gap_and_candidate_analysis" in selected:
        run_task_2_gap_and_candidate_analysis()
    if "toy_optimization" in selected:
        run_task_3_pedagogical_toy_optimization(mode=mode)
    if "optional_invrs_gym" in selected:
        run_task_4_optional_invrs_gym()

    if args.task is None or args.plot:
        plot_all()

    return 0


if __name__ == "__main__":
    raise SystemExit(main())